In [14]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LocalSparkTest")
    .master("local[1]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "1")
    .getOrCreate()
)
def runsql(query):
    spark.sql(query).show(truncate=False)

In [4]:
#21Given event logs (user_id, timestamp), sessionize events with a 30-minute inactivity timeout.


In [2]:
from pyspark.sql.functions import col, to_timestamp

data = [
    (1, "2026-01-01 09:00:00"),
    (1, "2026-01-01 09:10:00"),
    (1, "2026-01-01 09:25:00"),
    (1, "2026-01-01 10:10:00"),  # gap 45 min -> new session
    (1, "2026-01-01 10:20:00"),

    (2, "2026-01-01 08:00:00"),
    (2, "2026-01-01 08:20:00"),
    (2, "2026-01-01 09:00:00"),  # gap 40 min -> new session
    (2, "2026-01-01 09:10:00"),

    (3, "2026-01-01 12:00:00")
]

columns = ["user_id", "event_time"]

events_df = spark.createDataFrame(data, columns)

events_df = events_df.withColumn(
    "event_time",
    to_timestamp(col("event_time"))
)

events_df.createOrReplaceTempView("events")

events_df.show(truncate=False)

+-------+-------------------+
|user_id|event_time         |
+-------+-------------------+
|1      |2026-01-01 09:00:00|
|1      |2026-01-01 09:10:00|
|1      |2026-01-01 09:25:00|
|1      |2026-01-01 10:10:00|
|1      |2026-01-01 10:20:00|
|2      |2026-01-01 08:00:00|
|2      |2026-01-01 08:20:00|
|2      |2026-01-01 09:00:00|
|2      |2026-01-01 09:10:00|
|3      |2026-01-01 12:00:00|
+-------+-------------------+



In [29]:
runsql(""" 

with prev_events as (
select *, lead(event_time) over (partition by user_id order by event_time) next_event_time,
lag(event_time) over (partition by user_id order by event_time) pre_event_time

from events )
select * , pre_event_time,next_event_time
   ,case when next_event_time-event_time <=INTERVAL '0 00:30:00' DAY TO SECOND Then 'nex_same' else 'nex_not_same' end  as Nex_same

   ,case when event_time-pre_event_time <=INTERVAL '0 00:30:00' DAY TO SECOND Then 'pre_same' else 'pre_not_same' end  as Nex_same
from prev_events
""")

+-------+-------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+
|user_id|event_time         |next_event_time    |pre_event_time     |pre_event_time     |next_event_time    |Nex_same    |Nex_same    |
+-------+-------------------+-------------------+-------------------+-------------------+-------------------+------------+------------+
|1      |2026-01-01 09:00:00|2026-01-01 09:10:00|NULL               |NULL               |2026-01-01 09:10:00|nex_same    |pre_not_same|
|1      |2026-01-01 09:10:00|2026-01-01 09:25:00|2026-01-01 09:00:00|2026-01-01 09:00:00|2026-01-01 09:25:00|nex_same    |pre_same    |
|1      |2026-01-01 09:25:00|2026-01-01 10:10:00|2026-01-01 09:10:00|2026-01-01 09:10:00|2026-01-01 10:10:00|nex_not_same|pre_same    |
|1      |2026-01-01 10:10:00|2026-01-01 10:20:00|2026-01-01 09:25:00|2026-01-01 09:25:00|2026-01-01 10:20:00|nex_same    |pre_not_same|
|1      |2026-01-01 10:20:00|NULL               

In [30]:
sessionized_df = spark.sql("""
WITH ordered_events AS (
    SELECT
        user_id,
        event_time,

        LAG(event_time) OVER (
            PARTITION BY user_id
            ORDER BY event_time
        ) AS previous_event_time

    FROM events
),

flagged_events AS (
    SELECT
        user_id,
        event_time,
        previous_event_time,

        CASE
            WHEN previous_event_time IS NULL THEN 1
            WHEN UNIX_TIMESTAMP(event_time) - UNIX_TIMESTAMP(previous_event_time) > 30 * 60 THEN 1
            ELSE 0
        END AS new_session_flag

    FROM ordered_events
),

session_numbered AS (
    SELECT
        user_id,
        event_time,
        previous_event_time,
        new_session_flag,

        SUM(new_session_flag) OVER (
            PARTITION BY user_id
            ORDER BY event_time
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS session_number

    FROM flagged_events
)

SELECT
    user_id,
    event_time,
    previous_event_time,
    new_session_flag,
    session_number,
    CONCAT(user_id, '_', session_number) AS session_id
FROM session_numbered
ORDER BY user_id, event_time
""")

sessionized_df.show(truncate=False)

+-------+-------------------+-------------------+----------------+--------------+----------+
|user_id|event_time         |previous_event_time|new_session_flag|session_number|session_id|
+-------+-------------------+-------------------+----------------+--------------+----------+
|1      |2026-01-01 09:00:00|NULL               |1               |1             |1_1       |
|1      |2026-01-01 09:10:00|2026-01-01 09:00:00|0               |1             |1_1       |
|1      |2026-01-01 09:25:00|2026-01-01 09:10:00|0               |1             |1_1       |
|1      |2026-01-01 10:10:00|2026-01-01 09:25:00|1               |2             |1_2       |
|1      |2026-01-01 10:20:00|2026-01-01 10:10:00|0               |2             |1_2       |
|2      |2026-01-01 08:00:00|NULL               |1               |1             |2_1       |
|2      |2026-01-01 08:20:00|2026-01-01 08:00:00|0               |1             |2_1       |
|2      |2026-01-01 09:00:00|2026-01-01 08:20:00|1               |2   

In [37]:
runsql("""

WITH ordered_events AS (
    SELECT
        user_id,
        event_time,

        LAG(event_time) OVER (
            PARTITION BY user_id
            ORDER BY event_time
        ) AS previous_event_time

    FROM events
),

flagged_events AS (
    SELECT
        user_id,
        event_time,
        previous_event_time,

        CASE
            WHEN previous_event_time IS NULL THEN 1
            WHEN UNIX_TIMESTAMP(event_time) - UNIX_TIMESTAMP(previous_event_time) > 30 * 60 THEN 1
            ELSE 0
        END AS new_session_flag

    FROM ordered_events
)



Select * from flagged_events

"""
)


+-------+-------------------+-------------------+----------------+
|user_id|event_time         |previous_event_time|new_session_flag|
+-------+-------------------+-------------------+----------------+
|1      |2026-01-01 09:00:00|NULL               |1               |
|1      |2026-01-01 09:10:00|2026-01-01 09:00:00|0               |
|1      |2026-01-01 09:25:00|2026-01-01 09:10:00|0               |
|1      |2026-01-01 10:10:00|2026-01-01 09:25:00|1               |
|1      |2026-01-01 10:20:00|2026-01-01 10:10:00|0               |
|2      |2026-01-01 08:00:00|NULL               |1               |
|2      |2026-01-01 08:20:00|2026-01-01 08:00:00|0               |
|2      |2026-01-01 09:00:00|2026-01-01 08:20:00|1               |
|2      |2026-01-01 09:10:00|2026-01-01 09:00:00|0               |
|3      |2026-01-01 12:00:00|NULL               |1               |
+-------+-------------------+-------------------+----------------+



In [39]:
runsql("""

WITH ordered_events AS (
    SELECT
        user_id,
        event_time,

        LAG(event_time) OVER (
            PARTITION BY user_id
            ORDER BY event_time
        ) AS previous_event_time

    FROM events
),

flagged_events AS (
    SELECT
        user_id,
        event_time,
        previous_event_time,

        CASE
            WHEN previous_event_time IS NULL THEN 1
            WHEN UNIX_TIMESTAMP(event_time) - UNIX_TIMESTAMP(previous_event_time) > 30 * 60 THEN 1
            ELSE 0
        END AS new_session_flag

    FROM ordered_events
)
,
session_numbered AS (
    SELECT
        user_id,
        event_time,
        previous_event_time,
        new_session_flag,

        SUM(new_session_flag) OVER (
            PARTITION BY user_id
            ORDER BY event_time
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS session_number

    FROM flagged_events
)

Select * from session_numbered

"""
)


+-------+-------------------+-------------------+----------------+--------------+
|user_id|event_time         |previous_event_time|new_session_flag|session_number|
+-------+-------------------+-------------------+----------------+--------------+
|1      |2026-01-01 09:00:00|NULL               |1               |1             |
|1      |2026-01-01 09:10:00|2026-01-01 09:00:00|0               |1             |
|1      |2026-01-01 09:25:00|2026-01-01 09:10:00|0               |1             |
|1      |2026-01-01 10:10:00|2026-01-01 09:25:00|1               |2             |
|1      |2026-01-01 10:20:00|2026-01-01 10:10:00|0               |2             |
|2      |2026-01-01 08:00:00|NULL               |1               |1             |
|2      |2026-01-01 08:20:00|2026-01-01 08:00:00|0               |1             |
|2      |2026-01-01 09:00:00|2026-01-01 08:20:00|1               |2             |
|2      |2026-01-01 09:10:00|2026-01-01 09:00:00|0               |2             |
|3      |2026-01

In [43]:
runsql("""

WITH ordered_events AS (
    SELECT
        user_id,
        event_time,

        LAG(event_time) OVER (
            PARTITION BY user_id
            ORDER BY event_time
        ) AS previous_event_time

    FROM events
),

flagged_events AS (
    SELECT
        user_id,
        event_time,
        previous_event_time,

        CASE
            WHEN previous_event_time IS NULL THEN 1
            WHEN UNIX_TIMESTAMP(event_time) - UNIX_TIMESTAMP(previous_event_time) > 30 * 60 THEN 1
            ELSE 0
        END AS new_session_flag

    FROM ordered_events
)
,
session_numbered AS (
    SELECT
        user_id,
        event_time,
        previous_event_time,
        new_session_flag,

        SUM(new_session_flag) OVER (
            PARTITION BY user_id
            ORDER BY event_time
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS session_number

    FROM flagged_events
)
SELECT
    user_id,
    event_time,
    previous_event_time,
    new_session_flag,
    session_number,
    CONCAT(user_id, '_', session_number) AS session_id
FROM session_numbered
ORDER BY user_id, event_time

"""
)


+-------+-------------------+-------------------+----------------+--------------+----------+
|user_id|event_time         |previous_event_time|new_session_flag|session_number|session_id|
+-------+-------------------+-------------------+----------------+--------------+----------+
|1      |2026-01-01 09:00:00|NULL               |1               |1             |1_1       |
|1      |2026-01-01 09:10:00|2026-01-01 09:00:00|0               |1             |1_1       |
|1      |2026-01-01 09:25:00|2026-01-01 09:10:00|0               |1             |1_1       |
|1      |2026-01-01 10:10:00|2026-01-01 09:25:00|1               |2             |1_2       |
|1      |2026-01-01 10:20:00|2026-01-01 10:10:00|0               |2             |1_2       |
|2      |2026-01-01 08:00:00|NULL               |1               |1             |2_1       |
|2      |2026-01-01 08:20:00|2026-01-01 08:00:00|0               |1             |2_1       |
|2      |2026-01-01 09:00:00|2026-01-01 08:20:00|1               |2   

In [44]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lag, unix_timestamp, when, sum as spark_sum, concat_ws

window_spec = Window.partitionBy("user_id").orderBy("event_time")

session_window = (
    Window
    .partitionBy("user_id")
    .orderBy("event_time")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

sessionized_df = (
    events_df
    .withColumn("previous_event_time", lag("event_time").over(window_spec))
    .withColumn(
        "new_session_flag",
        when(col("previous_event_time").isNull(), 1)
        .when(
            unix_timestamp("event_time") - unix_timestamp("previous_event_time") > 30 * 60,
            1
        )
        .otherwise(0)
    )
    .withColumn(
        "session_number",
        spark_sum("new_session_flag").over(session_window)
    )
    .withColumn(
        "session_id",
        concat_ws("_", col("user_id"), col("session_number"))
    )
    .orderBy("user_id", "event_time")
)

sessionized_df.show(truncate=False)

+-------+-------------------+-------------------+----------------+--------------+----------+
|user_id|event_time         |previous_event_time|new_session_flag|session_number|session_id|
+-------+-------------------+-------------------+----------------+--------------+----------+
|1      |2026-01-01 09:00:00|NULL               |1               |1             |1_1       |
|1      |2026-01-01 09:10:00|2026-01-01 09:00:00|0               |1             |1_1       |
|1      |2026-01-01 09:25:00|2026-01-01 09:10:00|0               |1             |1_1       |
|1      |2026-01-01 10:10:00|2026-01-01 09:25:00|1               |2             |1_2       |
|1      |2026-01-01 10:20:00|2026-01-01 10:10:00|0               |2             |1_2       |
|2      |2026-01-01 08:00:00|NULL               |1               |1             |2_1       |
|2      |2026-01-01 08:20:00|2026-01-01 08:00:00|0               |1             |2_1       |
|2      |2026-01-01 09:00:00|2026-01-01 08:20:00|1               |2   

In [45]:
#24 Find users active for 3+ consecutive days.


In [47]:
from pyspark.sql.functions import col, to_date

data = [
    (1, "2026-01-01"),
    (1, "2026-01-02"),
    (1, "2026-01-03"),  # user 1 active 3 consecutive days
    (1, "2026-01-05"),

    (2, "2026-01-01"),
    (2, "2026-01-03"),
    (2, "2026-01-04"),  # only 2 consecutive days

    (3, "2026-01-10"),
    (3, "2026-01-11"),
    (3, "2026-01-12"),
    (3, "2026-01-13"),  # user 3 active 4 consecutive days

    (4, "2026-02-01"),
    (4, "2026-02-02"),
    (4, "2026-02-04")
]

columns = ["user_id", "activity_date"]

activity_df = spark.createDataFrame(data, columns)

activity_df = activity_df.withColumn(
    "activity_date",
    to_date(col("activity_date"))
)

activity_df.createOrReplaceTempView("user_activity")

activity_df.show()

+-------+-------------+
|user_id|activity_date|
+-------+-------------+
|      1|   2026-01-01|
|      1|   2026-01-02|
|      1|   2026-01-03|
|      1|   2026-01-05|
|      2|   2026-01-01|
|      2|   2026-01-03|
|      2|   2026-01-04|
|      3|   2026-01-10|
|      3|   2026-01-11|
|      3|   2026-01-12|
|      3|   2026-01-13|
|      4|   2026-02-01|
|      4|   2026-02-02|
|      4|   2026-02-04|
+-------+-------------+



In [75]:
runsql(

    """
    select * from user_activity as ua1
    INNER JOIN user_activity as ua2 on ua1.user_id=ua2.user_id and ua1.activity_date=DATE_SUB(ua1.activity_date,1)
    INNER JOIN user_activity as ua3 on ua1.user_id=ua3.user_id and ua1.activity_date=DATE_SUB(ua1.activity_date,2)

    """
)

+-------+-------------+-------+-------------+-------+-------------+
|user_id|activity_date|user_id|activity_date|user_id|activity_date|
+-------+-------------+-------+-------------+-------+-------------+
+-------+-------------+-------+-------------+-------+-------------+



In [74]:
runsql(

    """
       select *,DATE_SUB(ua1.activity_date,1) as date_diff
       from user_activity as ua1

    """
)

+-------+-------------+----------+
|user_id|activity_date|date_diff |
+-------+-------------+----------+
|1      |2026-01-01   |2025-12-31|
|1      |2026-01-02   |2026-01-01|
|1      |2026-01-03   |2026-01-02|
|1      |2026-01-05   |2026-01-04|
|2      |2026-01-01   |2025-12-31|
|2      |2026-01-03   |2026-01-02|
|2      |2026-01-04   |2026-01-03|
|3      |2026-01-10   |2026-01-09|
|3      |2026-01-11   |2026-01-10|
|3      |2026-01-12   |2026-01-11|
|3      |2026-01-13   |2026-01-12|
|4      |2026-02-01   |2026-01-31|
|4      |2026-02-02   |2026-02-01|
|4      |2026-02-04   |2026-02-03|
+-------+-------------+----------+



In [76]:
result_df = spark.sql("""
WITH distinct_activity AS (
    SELECT DISTINCT
        user_id,
        activity_date
    FROM user_activity
),

numbered AS (
    SELECT
        user_id,
        activity_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY activity_date
        ) AS rn
    FROM distinct_activity
),

grouped AS (
    SELECT
        user_id,
        activity_date,
        DATE_SUB(activity_date, rn) AS island_key
    FROM numbered
),

streaks AS (
    SELECT
        user_id,
        MIN(activity_date) AS streak_start_date,
        MAX(activity_date) AS streak_end_date,
        COUNT(*) AS consecutive_days
    FROM grouped
    GROUP BY
        user_id,
        island_key
)

SELECT
    user_id,
    streak_start_date,
    streak_end_date,
    consecutive_days
FROM streaks
WHERE consecutive_days >= 3
ORDER BY user_id, streak_start_date
""")

result_df.show()

+-------+-----------------+---------------+----------------+
|user_id|streak_start_date|streak_end_date|consecutive_days|
+-------+-----------------+---------------+----------------+
|      1|       2026-01-01|     2026-01-03|               3|
|      3|       2026-01-10|     2026-01-13|               4|
+-------+-----------------+---------------+----------------+



In [79]:
users_df = spark.sql("""
WITH distinct_activity AS (
    SELECT DISTINCT
        user_id,
        activity_date
    FROM user_activity
),

numbered AS (
    SELECT
        user_id,
        activity_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY activity_date
        ) AS rn
    FROM distinct_activity
),

grouped AS (
    SELECT
        user_id,
        activity_date,
        DATE_SUB(activity_date, rn) AS island_key
    FROM numbered
),

streaks AS (
    SELECT
        user_id,
        COUNT(*) AS consecutive_days
    FROM grouped
    GROUP BY
        user_id,
        island_key
    HAVING COUNT(*) >= 3
)

SELECT DISTINCT
    user_id
FROM streaks
ORDER BY user_id
""")

users_df.show()

+-------+
|user_id|
+-------+
|      1|
|      3|
+-------+



In [92]:
users_df = spark.sql("""
WITH distinct_activity AS (
    SELECT 
        user_id,
        activity_date
    FROM user_activity
    GROUP BY   user_id,
        activity_date
),
numbered AS (
    SELECT
        user_id,
        activity_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY activity_date
        ) AS rn
    FROM distinct_activity
),
grouped AS (
    SELECT
        user_id,
        activity_date,
        DATE_SUB(activity_date, rn) AS island_key
    FROM numbered
),
streaks AS (
    SELECT
        user_id,
        COUNT(*) AS consecutive_days
    FROM grouped
    GROUP BY
        user_id,
        island_key
    HAVING COUNT(*) >= 3
)
SELECT DISTINCT
    user_id
FROM streaks
ORDER BY user_id
""")
users_df.show()

+-------+
|user_id|
+-------+
|      1|
|      3|
+-------+



In [94]:
runsql("""
WITH activity AS (
    SELECT DISTINCT
        user_id,
        activity_date
    FROM user_activity
)
SELECT DISTINCT
    a1.user_id,
    a1.activity_date AS start_date,
    a2.activity_date AS second_day,
    a3.activity_date AS third_day
FROM activity a1
JOIN activity a2
    ON a1.user_id = a2.user_id
   AND a2.activity_date = DATE_ADD(a1.activity_date, 1)
JOIN activity a3
    ON a1.user_id = a3.user_id
   AND a3.activity_date = DATE_ADD(a1.activity_date, 2)
ORDER BY
    a1.user_id,
    start_date
""")

+-------+----------+----------+----------+
|user_id|start_date|second_day|third_day |
+-------+----------+----------+----------+
|1      |2026-01-01|2026-01-02|2026-01-03|
|3      |2026-01-10|2026-01-11|2026-01-12|
|3      |2026-01-11|2026-01-12|2026-01-13|
+-------+----------+----------+----------+



In [96]:
# 25.Find the longest streak of consecutive daily logins per user.


In [97]:
from pyspark.sql.functions import col, to_date

data = [
    (1, "2026-01-01"),
    (1, "2026-01-02"),
    (1, "2026-01-03"),
    (1, "2026-01-05"),
    (1, "2026-01-06"),

    (2, "2026-01-01"),
    (2, "2026-01-03"),
    (2, "2026-01-04"),
    (2, "2026-01-05"),

    (3, "2026-02-10"),
    (3, "2026-02-11"),
    (3, "2026-02-13")
]

columns = ["user_id", "login_date"]

logins_df = spark.createDataFrame(data, columns)

logins_df = logins_df.withColumn(
    "login_date",
    to_date(col("login_date"))
)

logins_df.createOrReplaceTempView("user_logins")

logins_df.show()

+-------+----------+
|user_id|login_date|
+-------+----------+
|      1|2026-01-01|
|      1|2026-01-02|
|      1|2026-01-03|
|      1|2026-01-05|
|      1|2026-01-06|
|      2|2026-01-01|
|      2|2026-01-03|
|      2|2026-01-04|
|      2|2026-01-05|
|      3|2026-02-10|
|      3|2026-02-11|
|      3|2026-02-13|
+-------+----------+



In [99]:
result_df = spark.sql("""
WITH distinct_logins AS (
    SELECT DISTINCT
        user_id,
        login_date
    FROM user_logins
),
numbered AS (
    SELECT
        user_id,
        login_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY login_date
        ) AS rn
    FROM distinct_logins
),

grouped AS (
    SELECT
        user_id,
        login_date,
        DATE_SUB(login_date, rn) AS island_key
    FROM numbered
),

streaks AS (
    SELECT
        user_id,
        MIN(login_date) AS streak_start_date,
        MAX(login_date) AS streak_end_date,
        COUNT(*) AS streak_days
    FROM grouped
    GROUP BY
        user_id,
        island_key
),

ranked_streaks AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY streak_days DESC, streak_start_date ASC
        ) AS rn
    FROM streaks
)

SELECT
    user_id,
    streak_start_date,
    streak_end_date,
    streak_days AS longest_streak_days
FROM ranked_streaks
WHERE rn = 1
ORDER BY user_id
""")

result_df.show()

+-------+-----------------+---------------+-------------------+
|user_id|streak_start_date|streak_end_date|longest_streak_days|
+-------+-----------------+---------------+-------------------+
|      1|       2026-01-01|     2026-01-03|                  3|
|      2|       2026-01-03|     2026-01-05|                  3|
|      3|       2026-02-10|     2026-02-11|                  2|
+-------+-----------------+---------------+-------------------+



In [100]:
result_df = spark.sql("""
WITH distinct_logins AS (
    SELECT DISTINCT
        user_id,
        login_date
    FROM user_logins
),

numbered AS (
    SELECT
        user_id,
        login_date,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY login_date
        ) AS rn
    FROM distinct_logins
),

grouped AS (
    SELECT
        user_id,
        login_date,
        DATE_SUB(login_date, rn) AS island_key
    FROM numbered
),

streaks AS (
    SELECT
        user_id,
        MIN(login_date) AS streak_start_date,
        MAX(login_date) AS streak_end_date,
        COUNT(*) AS streak_days
    FROM grouped
    GROUP BY
        user_id,
        island_key
),

ranked_streaks AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY streak_days DESC, streak_start_date ASC
        ) AS rn
    FROM streaks
)

SELECT
    user_id,
    streak_start_date,
    streak_end_date,
    streak_days AS longest_streak_days
FROM ranked_streaks
WHERE rn = 1
ORDER BY user_id
""")

result_df.show()

+-------+-----------------+---------------+-------------------+
|user_id|streak_start_date|streak_end_date|longest_streak_days|
+-------+-----------------+---------------+-------------------+
|      1|       2026-01-01|     2026-01-03|                  3|
|      2|       2026-01-03|     2026-01-05|                  3|
|      3|       2026-02-10|     2026-02-11|                  2|
+-------+-----------------+---------------+-------------------+



In [101]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, date_sub
from pyspark.sql.functions import min as spark_min, max as spark_max, count

distinct_logins_df = logins_df.select(
    "user_id",
    "login_date"
).distinct()

window_spec = Window.partitionBy("user_id").orderBy("login_date")

numbered_df = distinct_logins_df.withColumn(
    "rn",
    row_number().over(window_spec)
)

grouped_df = numbered_df.withColumn(
    "island_key",
    date_sub(col("login_date"), col("rn").cast("int"))
)

streaks_df = (
    grouped_df
    .groupBy("user_id", "island_key")
    .agg(
        spark_min("login_date").alias("streak_start_date"),
        spark_max("login_date").alias("streak_end_date"),
        count("*").alias("streak_days")
    )
)

rank_window = Window.partitionBy("user_id").orderBy(
    col("streak_days").desc(),
    col("streak_start_date").asc()
)

result_df = (
    streaks_df
    .withColumn("rn", row_number().over(rank_window))
    .filter(col("rn") == 1)
    .select(
        "user_id",
        "streak_start_date",
        "streak_end_date",
        col("streak_days").alias("longest_streak_days")
    )
    .orderBy("user_id")
)

result_df.show()

+-------+-----------------+---------------+-------------------+
|user_id|streak_start_date|streak_end_date|longest_streak_days|
+-------+-----------------+---------------+-------------------+
|      1|       2026-01-01|     2026-01-03|                  3|
|      2|       2026-01-03|     2026-01-05|                  3|
|      3|       2026-02-10|     2026-02-11|                  2|
+-------+-----------------+---------------+-------------------+



In [103]:
#26.Identify gaps in a sequence of IDs or dates.


In [104]:
data = [
    (1,),
    (2,),
    (3,),
    (7,),
    (8,),
    (10,),
    (11,),
    (15,)
]

columns = ["id"]

ids_df = spark.createDataFrame(data, columns)

ids_df.createOrReplaceTempView("ids")

ids_df.show()

+---+
| id|
+---+
|  1|
|  2|
|  3|
|  7|
|  8|
| 10|
| 11|
| 15|
+---+



In [105]:
data = [
    (1,),
    (2,),
    (3,),
    (7,),
    (8,),
    (10,),
    (11,),
    (15,)
]

columns = ["id"]

ids_df = spark.createDataFrame(data, columns)

ids_df.createOrReplaceTempView("ids")

ids_df.show()

+---+
| id|
+---+
|  1|
|  2|
|  3|
|  7|
|  8|
| 10|
| 11|
| 15|
+---+



In [107]:
result_df = spark.sql("""
WITH ordered_ids AS (
    SELECT
        id,
        LAG(id) OVER (ORDER BY id) AS previous_id
    FROM ids
),

gaps AS (
    SELECT
        previous_id + 1 AS gap_start,
        id - 1 AS gap_end
    FROM ordered_ids
    WHERE previous_id IS NOT NULL AND id > previous_id + 1
)

SELECT
    gap_start,
    gap_end,
    gap_end - gap_start + 1 AS missing_count
FROM gaps
ORDER BY gap_start
""")

result_df.show()

+---------+-------+-------------+
|gap_start|gap_end|missing_count|
+---------+-------+-------------+
|        4|      6|            3|
|        9|      9|            1|
|       12|     14|            3|
+---------+-------+-------------+



26/06/13 10:39:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 10:39:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 10:39:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 10:39:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 10:39:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [110]:
result_df = spark.sql("""
WITH ordered_ids AS (
    SELECT
        id,
        LAG(id) OVER (ORDER BY id) AS previous_id
    FROM ids
),

gaps AS (
    SELECT
        previous_id + 1 AS gap_start,
        id - 1 AS gap_end
    FROM ordered_ids
    WHERE previous_id IS NOT NULL AND id > previous_id + 1
)

select * from gaps
""")

result_df.show()

+---------+-------+
|gap_start|gap_end|
+---------+-------+
|        4|      6|
|        9|      9|
|       12|     14|
+---------+-------+



26/06/13 10:42:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 10:42:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 10:42:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 10:42:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 10:42:16 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [111]:
result_df = spark.sql("""
WITH ordered_ids AS (
    SELECT
        id,
        LAG(id) OVER (
            ORDER BY id
        ) AS previous_id
    FROM ids
),

gaps AS (
    SELECT
        previous_id + 1 AS gap_start,
        id - 1 AS gap_end
    FROM ordered_ids
    WHERE previous_id IS NOT NULL
      AND id > previous_id + 1
)

SELECT
    gap_start,
    gap_end,
    gap_end - gap_start + 1 AS missing_count
FROM gaps
ORDER BY gap_start
""")

result_df.show()

+---------+-------+-------------+
|gap_start|gap_end|missing_count|
+---------+-------+-------------+
|        4|      6|            3|
|        9|      9|            1|
|       12|     14|            3|
+---------+-------+-------------+



26/06/13 10:42:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 10:42:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 10:42:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 10:42:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/06/13 10:42:31 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
